In [ ]:
# ─────────────────────────────────────────────────────────────────
#  ▼  ÚNICA CELDA QUE NECESITAS EDITAR  ▼
# ─────────────────────────────────────────────────────────────────

DATA_ROOT   = r'C:\Users\frank\6to semestre\Python\OptimizacionM\data\musan'
OUTPUT_DIR  = r'C:\Users\frank\6to semestre\Python\OptimizacionM\nmf-audio-separation\outputs'

# Cuántos archivos analizar por categoría (None = todos)
# Con 30 es suficiente para el EDA; aumenta si quieres más estadístico
N_PER_CAT   = 30 

# Parámetros STFT (no cambiar para que coincidan con el modelo)
N_FFT       = 1024
HOP_LENGTH  = 512
SR_TARGET   = 22050

# ─────────────────────────────────────────────────────────────────

import os
from pathlib import Path

os.makedirs(OUTPUT_DIR, exist_ok=True)
CATEGORIES = ['music', 'noise', 'speech']

print(f'Datos  : {DATA_ROOT}')
print(f'Outputs: {OUTPUT_DIR}\n')

total = 0
for cat in CATEGORIES:
    p = Path(DATA_ROOT) / cat
    if p.exists():
        wavs = list(p.rglob('*.wav')) + list(p.rglob('*.flac'))
        mb   = sum(f.stat().st_size for f in wavs) / 1e6
        print(f'  {cat:8s}  {len(wavs):>6,} archivos   {mb:>8.1f} MB')
        total += len(wavs)
    else:
        print(f'  {cat:8s}  ❌ carpeta no encontrada: {p}')
print(f'  {"─"*42}')
print(f'  {"TOTAL":8s}  {total:>6,} archivos')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
import soundfile as sf
import warnings
from tqdm.notebook import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

PALETTE = {
    'music'  : '#4C72B0',
    'noise'  : '#DD8452',
    'speech' : '#55A868',
    'bg'     : '#F8F9FA',
    'grid'   : '#DEE2E6',
    'text'   : '#212529',
}
CAT_LABELS = {'music': 'Music', 'noise': 'Noise', 'speech': 'Speech'}

# ─────────────────────────────────────────────────────────────────

def scan_files(data_root, categories, n_per_cat=None, exts=('.wav','.flac','.mp3')):
    """Escanea carpetas y devuelve DataFrame con metadatos de archivos."""
    rows = []
    for cat in categories:
        path = Path(data_root) / cat
        if not path.exists():
            continue
        files = [f for f in path.rglob('*') if f.suffix.lower() in exts]
        if n_per_cat and len(files) > n_per_cat:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(files), n_per_cat, replace=False)
            files = [files[i] for i in sorted(idx)]
        for f in files:
            rows.append({
                'category'   : cat,
                'path'       : str(f),
                'filename'   : f.name,
                'size_bytes' : f.stat().st_size,
                'subdir'     : str(f.parent.relative_to(path)),
            })
    return pd.DataFrame(rows)


def analyze_file(path, sr_target=SR_TARGET, max_dur=30.0):
    """Extrae características acústicas y de espectrograma de un archivo."""
    try:
        info = sf.info(path)
        y, sr = librosa.load(path, sr=sr_target, mono=True, duration=max_dur)
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        return {
            'ok'                  : True,
            'sr_original'         : info.samplerate,
            'channels'            : info.channels,
            'duration_s'          : info.duration,
            'format'              : info.format,
            'rms_energy'          : float(np.sqrt(np.mean(y**2))),
            'zero_cross_rate'     : float(np.mean(librosa.feature.zero_crossing_rate(y))),
            'spectral_centroid_hz': float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))),
            'spectral_bandwidth'  : float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))),
            'spectral_rolloff_hz' : float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))),
            'stft_F'              : S.shape[0],
            'stft_T'              : S.shape[1],
            'stft_mean'           : float(S.mean()),
            'stft_max'            : float(S.max()),
        }
    except Exception as e:
        return {'ok': False, 'error': str(e)}


def ax_style(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor(PALETTE['bg'])
    ax.set_title(title, fontsize=9.5, fontweight='bold', pad=6, color=PALETTE['text'])
    ax.set_xlabel(xlabel, fontsize=8, color='#495057')
    ax.set_ylabel(ylabel, fontsize=8, color='#495057')
    ax.tick_params(labelsize=7.5, colors='#495057')
    ax.grid(True, color=PALETTE['grid'], linewidth=0.5, alpha=0.8, zorder=0)
    for sp in ax.spines.values(): sp.set_edgecolor(PALETTE['grid'])

print('Imports y funciones listas')         

 Celda 8 — Figura 3: Zero Crossing Rate (ZCR)

In [ ]:

fig3, axes = plt.subplots(1, len(cats), figsize=(20, 4), facecolor=PALETTE['bg'])
fig3.suptitle('Análisis de Tasa de Cruce por Cero (ZCR) por Categoría', 
              fontsize=12, fontweight='bold', y=1.05, color=PALETTE['text'])

for ax, cat, color, label in zip(axes, cats, colors, labels):
    # Tomar el primer archivo de cada categoría como muestra
    path = df_ok[df_ok['category']==cat]['path'].iloc[0]
    y, sr = librosa.load(path, sr=SR_TARGET, mono=True, duration=5.0)
    
    # Calcular ZCR
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    
    ax.plot(zcr, color=color, alpha=0.9)
    ax.set_title(f'ZCR: {label}', color=PALETTE['text'])
    ax.set_xlabel('Frames')
    ax.set_ylabel('Tasa de Cruces')
    ax.grid(color=PALETTE['grid'], linestyle='--', alpha=0.5)
    ax.set_facecolor(PALETTE['bg'])

plt.tight_layout()
plt.show()

Celda 9 — Figura 4: Distribución y Outliers (Histograma STFT)


In [ ]:

fig4, axes = plt.subplots(1, len(cats), figsize=(20, 4), facecolor=PALETTE['bg'])
fig4.suptitle('Distribución de Magnitudes Espectrales (Matriz X) y Outliers', 
              fontsize=12, fontweight='bold', y=1.05, color=PALETTE['text'])

for ax, cat, color, label in zip(axes, cats, colors, labels):
    path = df_ok[df_ok['category']==cat]['path'].iloc[0]
    y, sr = librosa.load(path, sr=SR_TARGET, mono=True, duration=5.0)
    
    # Calcular Espectrograma (Matriz X)
    X_matrix = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
    
    # Aplanar la matriz para ver la distribución de todos los valores
    ax.hist(X_matrix.flatten(), bins=50, color=color, alpha=0.7, log=True)
    ax.set_title(f'Distribución STFT: {label}', color=PALETTE['text'])
    ax.set_xlabel('Magnitud')
    ax.set_ylabel('Frecuencia (Log Scale)')
    ax.grid(color=PALETTE['grid'], linestyle='--', alpha=0.5)
    ax.set_facecolor(PALETTE['bg'])

plt.tight_layout()
plt.show()